# Итоговый pipeline решения

Загрузка -> Тайлинг -> Детекция + Положение -> Склейка -> Трекинг -> Итоговые результаты

In [2]:
import sys
import os
from pathlib import Path
import random, shutil, math
from dataclasses import dataclass

import numpy as np
import pandas as pd
import yaml

import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch, Circle
from PIL import Image

from tqdm.auto import tqdm
import torch
from ultralytics import YOLO

import supervision as sv
from trackers import OCSORTTracker

from sklearn.decomposition import PCA

%matplotlib inline

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


device: 0


In [1]:
!pip install ultralytics supervision trackers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.6/273.6 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 5.0 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pathlib import Path
dataset_dir = Path("/content/drive/MyDrive/bee_dataset")

frames_dir = dataset_dir / "frames"
positions_dir = dataset_dir / "positions"

In [5]:
# config
PROJECTDIR = Path("/content/drive/MyDrive/bee_dataset")
FRAMESDIR = PROJECTDIR / "frames"
ANNOTATIONSDIR = PROJECTDIR / "positions"
YOLODATASETDIR = PROJECTDIR / "yolo-bees-pose-dataset"
RUNSDIR = PROJECTDIR / "runs"
TRACKOUTPUTDIR = PROJECTDIR / "tracking-outputs"


In [6]:
# инициализация трекера
# ссылка на источник: https://trackers.roboflow.com/develop/trackers/ocsort/#how-does-oc-sort-work
# описание парамертов приведено из источника
tracker = OCSORTTracker(
    lost_track_buffer = 5,             # Frames to keep an unmatched track alive before deletion
    minimum_consecutive_frames = 3,    # Consecutive detections required to confirm a new track
    minimum_iou_threshold = 0.3,       # Minimum IoU to accept a track-detection match
    delta_t= 3                         # Frame gap for OCM velocity re-estimation after occlusion
)

In [7]:
def yolo_result_to_sv_detections(result, conf_thres=0.25):
    boxes = result.boxes
    if boxes is None or boxes.xyxy is None or len(boxes) == 0:
        return sv.Detections.empty()

    xyxy = boxes.xyxy.cpu().numpy()
    conf = boxes.conf.cpu().numpy() if boxes.conf is not None else np.ones(len(xyxy), dtype=np.float32)
    cls = boxes.cls.cpu().numpy() if boxes.cls is not None else np.zeros(len(xyxy), dtype=np.int32)

    keep = conf >= conf_thres
    xyxy = xyxy[keep]
    conf = conf[keep]
    cls = cls[keep]

    if len(xyxy) == 0:
        return sv.Detections.empty()

    return sv.Detections(
        xyxy=xyxy.astype(np.float32),
        confidence=conf.astype(np.float32),
        class_id=cls.astype(np.int32),
    )

In [8]:
def update_ocsort_from_result(result, conf_thres=0.25):
    detections = yolo_result_to_sv_detections(result, conf_thres=conf_thres)
    tracked = tracker.update(detections=detections)
    return detections, tracked

In [9]:
def track_with_ocsort_supervision(model, source, conf=0.25, imgsz=640, device=None):
    det_rows = []
    track_rows = []

    results = model.predict(source=source, stream=True, conf=conf, imgsz=imgsz, device=device, verbose=False)

    for frame_idx, r in enumerate(results):
        detections = yolo_result_to_sv_detections(r, conf_thres=conf)
        tracked = tracker.update(detections=detections)

        if len(detections) > 0:
            for j in range(len(detections)):
                det_rows.append({
                    "frame_idx": frame_idx,
                    "det_id": j,
                    "x1": float(detections.xyxy[j][0]),
                    "y1": float(detections.xyxy[j][1]),
                    "x2": float(detections.xyxy[j][2]),
                    "y2": float(detections.xyxy[j][3]),
                    "score": float(detections.confidence[j]) if detections.confidence is not None else np.nan,
                    "cls": int(detections.class_id[j]) if detections.class_id is not None else -1,
                })

        if len(tracked) > 0 and tracked.tracker_id is not None:
            for j in range(len(tracked)):
                track_rows.append({
                    "frame_idx": frame_idx,
                    "track_id": int(tracked.tracker_id[j]),
                    "x1": float(tracked.xyxy[j][0]),
                    "y1": float(tracked.xyxy[j][1]),
                    "x2": float(tracked.xyxy[j][2]),
                    "y2": float(tracked.xyxy[j][3]),
                    "score": float(tracked.confidence[j]) if tracked.confidence is not None else np.nan,
                    "cls": int(tracked.class_id[j]) if tracked.class_id is not None else -1,
                })

    det_df = pd.DataFrame(det_rows)
    track_df = pd.DataFrame(track_rows)
    return det_df, track_df

In [10]:
# визуализация на фреймах
def visualize_tracking_frames(source, track_df, max_frames=6, img_size=(16, 10)):
    source = Path(source)
    image_files = sorted(
        [p for p in source.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}]
    )[:max_frames]

    if len(image_files) == 0:
        print("Не найдено изображений в source.")
        return

    if track_df is None or len(track_df) == 0:
        print("track_df пустой.")
        return

    cols = 3
    rows = (len(image_files) + cols - 1) // cols
    plt.figure(figsize=img_size)

    for i, img_path in enumerate(image_files):
        frame_idx = i
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        frame_tracks = track_df[track_df["frame_idx"] == frame_idx].copy()

        if len(frame_tracks) > 0:
            detections = sv.Detections(
                xyxy=frame_tracks[["x1", "y1", "x2", "y2"]].values.astype(np.float32),
                confidence=frame_tracks["score"].values.astype(np.float32) if "score" in frame_tracks.columns else None,
                class_id=frame_tracks["cls"].values.astype(np.int32) if "cls" in frame_tracks.columns else np.zeros(len(frame_tracks), dtype=np.int32),
                tracker_id=frame_tracks["track_id"].values.astype(np.int32),
            )

            labels = [
                f"ID {int(tid)}"
                for tid in frame_tracks["track_id"].values
            ]

            annotated = img.copy()

            box_annotator = sv.BoxAnnotator(color=sv.Color(190, 50, 150)) # цвета в формате BGR для openCV
            label_annotator = sv.LabelAnnotator(color=sv.Color(190, 50, 150))

            annotated = box_annotator.annotate(scene=annotated, detections=detections)
            annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)

        else:
            annotated = img

        ax = plt.subplot(rows, cols, i + 1)
        ax.imshow(annotated)
        ax.set_title(f"Frame {frame_idx}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [11]:
# параметры
TRACK_SOURCE = PROJECTDIR / 'final_test'
TILE_SIZE = 640
OVERLAP = 0.2
CONF_THRES = 0.25
IOU_THRES = 0.5

In [13]:
import torchvision

PURPLE = (156, 102, 233)
YELLOW = (255, 255, 0)
GREEN = (0, 255, 0)
BLUEVIOLET = (138, 43, 226)
ARROW_MIN_DIST = 8


def tile_image(img, tile_size=640, overlap=0.2):
    h, w = img.shape[:2]
    step = max(1, int(tile_size * (1 - overlap)))
    tiles = []
    for y in range(0, h, step):
        for x in range(0, w, step):
            x2 = min(x + tile_size, w)
            y2 = min(y + tile_size, h)
            x1 = max(0, x2 - tile_size)
            y1 = max(0, y2 - tile_size)
            tile = img[y1:y2, x1:x2]
            tiles.append((tile, x1, y1))
            if x2 == w:
                break
        if y2 == h:
            break
    return tiles


def yolo_result_to_xyxy_conf_cls_kpts(result, conf_thres=0.25):
    boxes = result.boxes
    if boxes is None or boxes.xyxy is None or len(boxes) == 0:
        return np.empty((0, 6), dtype=np.float32), []

    xyxy = boxes.xyxy.cpu().numpy()
    conf = boxes.conf.cpu().numpy() if boxes.conf is not None else np.ones(len(xyxy), dtype=np.float32)
    cls = boxes.cls.cpu().numpy() if boxes.cls is not None else np.zeros(len(xyxy), dtype=np.float32)

    keep = conf >= conf_thres
    if keep.sum() == 0:
        return np.empty((0, 6), dtype=np.float32), []

    dets = np.column_stack([xyxy[keep], conf[keep], cls[keep]]).astype(np.float32)

    kpts_list = [None] * len(dets)
    if getattr(result, "keypoints", None) is not None:
        try:
            kdata = result.keypoints.data.cpu().numpy()[keep]
            kpts_list = [k.astype(np.float32) for k in kdata]
        except Exception:
            kpts_list = [None] * len(dets)

    return dets, kpts_list


def nms_keep_indices(dets, iou_thres=0.5):
    """Глобальный NMS по конкатенированным детекциям всех тайлов.
    Возвращает индексы боксов, которые надо оставить (убивает дубли в зонах перекрытия тайлов)."""
    if len(dets) == 0:
        return np.empty((0,), dtype=np.int64)
    boxes = torch.tensor(dets[:, :4], dtype=torch.float32)
    scores = torch.tensor(dets[:, 4], dtype=torch.float32)
    keep = torchvision.ops.nms(boxes, scores, float(iou_thres))
    return keep.cpu().numpy()


def _iou_one_vs_many(a, b):
    """IoU одного бокса a (4,) против массива боксов b (N,4)."""
    if len(b) == 0:
        return np.empty((0,), dtype=np.float32)
    ax1, ay1, ax2, ay2 = a[:4]
    bx1, by1, bx2, by2 = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    ix1 = np.maximum(ax1, bx1); iy1 = np.maximum(ay1, by1)
    ix2 = np.minimum(ax2, bx2); iy2 = np.minimum(ay2, by2)
    iw = np.clip(ix2 - ix1, 0, None); ih = np.clip(iy2 - iy1, 0, None)
    inter = iw * ih
    area_a = max((ax2 - ax1), 0) * max((ay2 - ay1), 0)
    area_b = np.clip(bx2 - bx1, 0, None) * np.clip(by2 - by1, 0, None)
    union = area_a + area_b - inter + 1e-6
    return inter / union


def match_tracked_to_dets(tracked_xyxy, merged_xyxy, iou_min=0.3):
    """Для каждого трек-бокса находит индекс ближайшей детекции (чтобы подтянуть кейпоинты). -1 если совпадения нет."""
    out = []
    merged_xyxy = np.asarray(merged_xyxy, dtype=np.float32)
    for tb in tracked_xyxy:
        if len(merged_xyxy) == 0:
            out.append(-1); continue
        ious = _iou_one_vs_many(np.asarray(tb, dtype=np.float32), merged_xyxy)
        j = int(np.argmax(ious))
        out.append(j if ious[j] >= iou_min else -1)
    return out


def track_frame_with_tiles(frame, model, tracker=None, conf=0.25, tile_size=640, overlap=0.2, iou_thres=0.5):
    tiles = tile_image(frame, tile_size=tile_size, overlap=overlap)
    all_dets = []
    all_kpts = []

    for tile, ox, oy in tiles:
        r = model.predict(tile, conf=conf, verbose=False)[0]
        dets, kpts_list = yolo_result_to_xyxy_conf_cls_kpts(r, conf_thres=conf)
        if len(dets) == 0:
            continue
        dets[:, [0, 2]] += ox
        dets[:, [1, 3]] += oy
        for di in range(len(dets)):
            all_dets.append(dets[di])
            k = kpts_list[di]
            if k is not None:
                k = k.copy(); k[:, 0] += ox; k[:, 1] += oy
            all_kpts.append(k)

    if len(all_dets) == 0:
        return np.empty((0, 6), dtype=np.float32), []

    merged = np.stack(all_dets, axis=0)
    keep = np.sort(nms_keep_indices(merged, iou_thres=iou_thres))
    merged = merged[keep]
    merged_kpts = [all_kpts[k] for k in keep]
    return merged, merged_kpts


def run_pipeline(source, model, tracker, conf=0.25, tile_size=640, overlap=0.2, iou_thres=0.5):
    source = Path(source)
    track_rows = []
    frames = []

    if source.is_dir():
        files = sorted([p for p in source.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}])

        for frame_idx, path in enumerate(files):
            frame = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)

            tiles = tile_image(frame, tile_size=tile_size, overlap=overlap)
            all_dets = []
            all_kpts = []

            for tile, ox, oy in tiles:
                r = model.predict(tile, conf=conf, verbose=False)[0]
                dets, kpts_list = yolo_result_to_xyxy_conf_cls_kpts(r, conf_thres=conf)
                if len(dets) == 0:
                    continue
                dets[:, [0, 2]] += ox
                dets[:, [1, 3]] += oy
                for di in range(len(dets)):
                    all_dets.append(dets[di])
                    k = kpts_list[di]
                    if k is not None:
                        k = k.copy(); k[:, 0] += ox; k[:, 1] += oy
                    all_kpts.append(k)

            if len(all_dets) == 0:
                frames.append((frame_idx, frame, np.empty((0, 6), dtype=np.float32), [], None))
                continue

            merged_dets = np.stack(all_dets, axis=0)

            keep = np.sort(nms_keep_indices(merged_dets, iou_thres=iou_thres))
            merged_dets = merged_dets[keep]
            merged_kpts = [all_kpts[k] for k in keep]

            detections = sv.Detections(
                xyxy=merged_dets[:, :4].astype(np.float32),
                confidence=merged_dets[:, 4].astype(np.float32),
                class_id=merged_dets[:, 5].astype(np.int32),
            )
            tracked = tracker.update(detections=detections)

            if tracked is not None and len(tracked) > 0 and tracked.tracker_id is not None:
                for k in range(len(tracked)):
                    track_rows.append({
                        "frame_idx": frame_idx,
                        "track_id": int(tracked.tracker_id[k]),
                        "x1": float(tracked.xyxy[k][0]),
                        "y1": float(tracked.xyxy[k][1]),
                        "x2": float(tracked.xyxy[k][2]),
                        "y2": float(tracked.xyxy[k][3]),
                        "score": float(tracked.confidence[k]) if tracked.confidence is not None else np.nan,
                    })

            frames.append((frame_idx, frame, merged_dets, merged_kpts, tracked))

    return pd.DataFrame(track_rows), frames


def _draw_circle_around_point(img, cx, cy, r=8, color=YELLOW, thickness=2):
    cv2.circle(img, (int(cx), int(cy)), r, color, thickness, lineType=cv2.LINE_AA)


def _draw_pose(img, kpts, color=YELLOW, radius=4):
    if kpts is None:
        return False
    pts = np.array(kpts, dtype=np.float32)
    if pts.ndim != 2 or pts.shape[1] < 2:
        return False
    vis = pts[:, 2] if pts.shape[1] > 2 else np.ones(len(pts), dtype=np.float32)
    for i, (x, y) in enumerate(pts[:, :2]):
        if vis[i] <= 0:
            continue
        cv2.circle(img, (int(x), int(y)), radius, color, -1, lineType=cv2.LINE_AA)
    for i in range(len(pts) - 1):
        if vis[i] > 0 and vis[i + 1] > 0:
            p1 = (int(pts[i, 0]), int(pts[i, 1]))
            p2 = (int(pts[i + 1, 0]), int(pts[i + 1, 1]))
            cv2.line(img, p1, p2, color, 2, lineType=cv2.LINE_AA)
    return True


def _valid_kpts(kpts):
    """Список видимых точек [(x,y), ...] в порядке [head, tail]."""
    if kpts is None:
        return []
    pts = np.asarray(kpts, dtype=np.float32)
    if pts.ndim != 2 or pts.shape[1] < 2:
        return []
    vis = pts[:, 2] if pts.shape[1] > 2 else np.ones(len(pts), dtype=np.float32)
    out = []
    for (x, y), v in zip(pts[:, :2], vis):
        if np.isnan(x) or np.isnan(y):
            continue
        if v > 0:
            out.append((float(x), float(y)))
    return out


def _kpt_orientation(kpts, min_dist=ARROW_MIN_DIST):
    """Единичный вектор ориентации (head-tail) для полностью видимой пчелы.
    Возвращает None, если точек < 2 или они ближе min_dist (пчела видна не полностью)."""
    valid = _valid_kpts(kpts)
    if len(valid) < 2:
        return None
    p0, p1 = valid[0], valid[1]   # p0=head, p1=tail
    dist = ((p0[0] - p1[0]) ** 2 + (p0[1] - p1[1]) ** 2) ** 0.5
    if dist < min_dist:
        return None
    v = np.array([p0[0] - p1[0], p0[1] - p1[1]], dtype=np.float32)
    n = np.linalg.norm(v)
    return v / n if n > 1e-6 else None


def _draw_pose_marker(img, kpts, color=YELLOW, min_dist=ARROW_MIN_DIST):
    """Разметка
      - 2 точки и расстояние >= min_dist -> жёлтые точки + стрелка '<-' на голову;
      - 2 точки и расстояние < min_dist  -> пчела видна не полностью -> кружок;
      - ровно 1 видимая точка             -> точка + маленький кружок;
      - нет видимых точек                 -> ничего (только бокс рисуется отдельно).
    """
    valid = _valid_kpts(kpts)
    if len(valid) == 0:
        return

    if len(valid) == 1:
        x, y = valid[0]
        cv2.circle(img, (int(x), int(y)), 3, color, -1, lineType=cv2.LINE_AA)
        cv2.circle(img, (int(x), int(y)), 8, color, 2, lineType=cv2.LINE_AA)
        return

    p0, p1 = valid[0], valid[1]   # p0=head, p1=tail
    dist = ((p0[0] - p1[0]) ** 2 + (p0[1] - p1[1]) ** 2) ** 0.5

    for x, y in valid:
        cv2.circle(img, (int(x), int(y)), 4, color, -1, lineType=cv2.LINE_AA)

    if dist >= min_dist:
        cv2.arrowedLine(img, (int(p1[0]), int(p1[1])), (int(p0[0]), int(p0[1])),
                        color, 2, cv2.LINE_AA, 0, 0.3)
    else:
        cv2.circle(img, (int(p0[0]), int(p0[1])), 15, color, 2, lineType=cv2.LINE_AA)


In [18]:
def draw_compass(img, direction, angle, polarization, position='top-right'):

    h, w = img.shape[:2]

    compass_size = min(w, h) // 12
    margin = 20

    if position == 'top-left':
        cx, cy = margin + compass_size, margin + compass_size
    elif position == 'top-right':
        cx, cy = w - margin - compass_size, margin + compass_size
    elif position == 'bottom-left':
        cx, cy = margin + compass_size, h - margin - compass_size
    else:
        cx, cy = w - margin - compass_size, h - margin - compass_size


    padding = 15
    rect_size = compass_size * 2 + padding * 2

    x1 = cx - compass_size - padding
    y1 = cy - compass_size - padding
    x2 = cx + compass_size + padding
    y2 = cy + compass_size + padding + 75

    cv2.rectangle(img, (x1+3, y1+3), (x2+3, y2+3), (0, 0, 0), -1)
    cv2.rectangle(img, (x1+3, y1+3), (x2+3, y2+3), (0, 0, 0), 1)

    cv2.rectangle(img, (x1, y1), (x2, y2), (255, 255, 255), -1)
    cv2.rectangle(img, (x1, y1), (x2, y2), (200, 200, 200), 1)

    cv2.circle(img, (cx, cy), compass_size, (240, 240, 240), -1)
    cv2.circle(img, (cx, cy), compass_size, (180, 180, 180), 1)
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.4

    for angle_deg in range(0, 360, 45):
        rad = np.radians(angle_deg)
        r1 = compass_size - 5
        r2 = compass_size - 12 if angle_deg % 90 == 0 else compass_size - 8
        x1d = cx + r1 * np.cos(rad)
        y1d = cy + r1 * np.sin(rad)
        x2d = cx + r2 * np.cos(rad)
        y2d = cy + r2 * np.sin(rad)
        cv2.line(img, (int(x1d), int(y1d)), (int(x2d), int(y2d)), (150, 150, 150), 1)

    arrow_length = compass_size - 10
    ex_swarm = cx + arrow_length * direction[0]
    ey_swarm = cy + arrow_length * direction[1]

    cv2.arrowedLine(
        img,
        (cx, cy),
        (int(ex_swarm), int(ey_swarm)),
        (255, 255, 255),
        4,
        tipLength=0.3
    )
    cv2.arrowedLine(
        img,
        (cx, cy),
        (int(ex_swarm), int(ey_swarm)),
        (255, 100, 0),
        2,
        tipLength=0.3
    )

    if len(direction) > 2 and (direction[2] != 0 or direction[3] != 0):
        ex_look = cx + arrow_length * direction[2]
        ey_look = cy + arrow_length * direction[3]

        cv2.arrowedLine(
            img,
            (cx, cy),
            (int(ex_look), int(ey_look)),
            (255, 255, 255),
            4,
            tipLength=0.3
        )
        cv2.arrowedLine(
            img,
            (cx, cy),
            (int(ex_look), int(ey_look)),
            (0, 0, 255),
            2,
            tipLength=0.3
        )

    cv2.circle(img, (cx, cy), 3, (100, 100, 100), -1)

    info_font_scale = 0.35
    info_y = cy + compass_size + 20

    cv2.putText(img, f"{angle:.0f}°", (cx - 18, info_y), font, info_font_scale, (50, 50, 50), 1)
    cv2.putText(img, f"P:{polarization:.2f}", (cx - 22, info_y + 15), font, info_font_scale, (50, 50, 50), 1)

    legend_y = info_y + 28

    cv2.arrowedLine(img, (cx - 35, legend_y), (cx - 20, legend_y), (255, 100, 0), 2, tipLength=0.3)
    cv2.putText(img, "Axis", (cx - 15, legend_y + 4), font, 0.3, (50, 50, 50), 1)

    cv2.arrowedLine(img, (cx + 15, legend_y), (cx + 30, legend_y), (0, 0, 255), 2, tipLength=0.3)
    cv2.putText(img, "Look", (cx + 35, legend_y + 4), font, 0.3, (50, 50, 50), 1)

In [15]:
def save_annotated_video(frames, out_path, fps=20, save_frames_dir=None):
    if len(frames) == 0:
        raise ValueError("frames пустой.")

    first_frame = frames[0][1].copy()
    h, w = first_frame.shape[:2]

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if save_frames_dir is not None:
        save_frames_dir = Path(save_frames_dir)
        save_frames_dir.mkdir(parents=True, exist_ok=True)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(out_path), fourcc, fps, (w, h))
    if not writer.isOpened():
        raise RuntimeError("VideoWriter не открылся")

    for item in frames:
        if len(item) == 5:
            frame_idx, frame, merged, kpts_list, tracked = item
        elif len(item) == 4:
            frame_idx, frame, merged, kpts_list = item
            tracked = None
        elif len(item) == 3:
            frame_idx, frame, merged = item
            kpts_list, tracked = [], None
        else:
            continue

        img = frame.copy()
        merged = np.asarray(merged) if merged is not None else np.empty((0, 6), dtype=np.float32)
        has_tracks = tracked is not None and len(tracked) > 0 and tracked.tracker_id is not None

        if has_tracks:
            t_xyxy = tracked.xyxy
            t_ids = tracked.tracker_id
            match_idx = match_tracked_to_dets(t_xyxy, merged[:, :4] if len(merged) else np.empty((0, 4)))
        else:
            t_xyxy, t_ids, match_idx = [], [], []

        centers = []
        orientation_vectors = []

        if has_tracks:
            for k in range(len(t_xyxy)):
                x1, y1, x2, y2 = t_xyxy[k][:4]
                centers.append([(x1 + x2) / 2, (y1 + y2) / 2])
                di = match_idx[k]
                if di >= 0 and kpts_list is not None and di < len(kpts_list):
                    vec = _kpt_orientation(kpts_list[di])
                    if vec is not None:
                        orientation_vectors.append(vec)
        else:
            for j, d in enumerate(merged):
                if len(d) >= 4:
                    x1, y1, x2, y2 = d[:4]
                    centers.append([(x1 + x2) / 2, (y1 + y2) / 2])
                    if kpts_list is not None and j < len(kpts_list):
                        vec = _kpt_orientation(kpts_list[j])
                        if vec is not None:
                            orientation_vectors.append(vec)

        if len(centers) >= 3:
            centers_np = np.asarray(centers, dtype=np.float32)
            pca = PCA(n_components=2)
            pca.fit(centers_np)
            swarm_axis = pca.components_[0]

            mean_look_vector = None
            if len(orientation_vectors) > 0:
                mean_dir = np.mean(orientation_vectors, axis=0)
                mean_norm = np.linalg.norm(mean_dir)
                if mean_norm > 1e-6:
                    mean_dir /= mean_norm
                    mean_look_vector = mean_dir
                    if np.dot(swarm_axis, mean_dir) < 0:
                        swarm_axis = -swarm_axis

            swarm_angle = np.degrees(np.arctan2(swarm_axis[1], swarm_axis[0]))
            polarization = 0.0
            if len(orientation_vectors) > 0:
                polarization = np.linalg.norm(np.mean(orientation_vectors, axis=0))

            if mean_look_vector is not None:
                compass_data = np.array([swarm_axis[0], swarm_axis[1],
                                         mean_look_vector[0], mean_look_vector[1]])
                draw_compass(img, compass_data, swarm_angle, polarization, 'bottom-right')
            else:
                draw_compass(img, np.array([swarm_axis[0], swarm_axis[1], 0, 0]),
                             swarm_angle, polarization, 'bottom-right')

        if has_tracks:
            for k in range(len(t_xyxy)):
                x1, y1, x2, y2 = map(int, t_xyxy[k][:4])
                tid = int(t_ids[k])
                cv2.rectangle(img, (x1, y1), (x2, y2), BLUEVIOLET, 2)
                if tid >= 0:
                    cv2.putText(img, f"ID {tid}", (x1 + 2, y1 + 12),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.4, BLUEVIOLET, 1)
                di = match_idx[k]
                if di >= 0 and kpts_list is not None and di < len(kpts_list):
                    _draw_pose_marker(img, kpts_list[di])
        else:
            for j, d in enumerate(merged):
                if len(d) >= 4:
                    x1, y1, x2, y2 = map(int, d[:4])
                    cv2.rectangle(img, (x1, y1), (x2, y2), BLUEVIOLET, 2)
                    if kpts_list is not None and j < len(kpts_list):
                        _draw_pose_marker(img, kpts_list[j])

        if save_frames_dir is not None:
            cv2.imwrite(str(save_frames_dir / f"{frame_idx:06d}.jpg"),
                        cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
        writer.write(cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

    writer.release()
    return str(out_path)


In [16]:
# запуск
# ВАЖНО: трекер хранит состояние между кадрами. Перед каждым полным прогоном (ячейка ниже)
# нужно создавать заново, иначе ID и буферы потянутся со старого запуска.
bestpt = Path("/content/drive/MyDrive/bee_dataset/runs/bee_pose_minimal_ocsort/weights/best.pt")
model = YOLO(str(bestpt if bestpt.exists() else "yolo11n-pose.pt"))


In [20]:
# сохранение результатов
tracker = OCSORTTracker(
    lost_track_buffer=3,
    frame_rate=15,
    minimum_consecutive_frames=1,
    minimum_iou_threshold=0.2,
    high_conf_det_threshold=0.1,
    delta_t=3,
)

track_df, frames = run_pipeline(
    source=TRACK_SOURCE,
    model=model,
    tracker=tracker,
    conf=0.3,
    tile_size=640,
    overlap=0.2,
    iou_thres=0.6,
)
print(len(frames))
print(track_df.head())
if len(track_df):
    print("уникальных track_id:", track_df["track_id"].nunique(),
          "| доля -1:", round((track_df["track_id"] == -1).mean(), 3))

frames_output_dir = "output/frames"
os.makedirs(frames_output_dir, exist_ok=True)

video_path = save_annotated_video(
    frames=frames,
    out_path="output/annotated_pose.mp4",
    fps=15,
    save_frames_dir=frames_output_dir
)
print("Saved video to:", video_path)
print("Saved frames to:", frames_output_dir)

# перекодирование в H.264 для воспроизведения
import subprocess, shutil

def reencode_h264(src_path, dst_path=None):
    src_path = str(src_path)
    if dst_path is None:
        dst_path = src_path.rsplit('.', 1)[0] + '_h264.mp4'
    if shutil.which('ffmpeg') is None:
        print('ffmpeg не найден — пропускаю перекодирование, исходный mp4v может не играться')
        return src_path
    cmd = [
        'ffmpeg', '-y', '-i', src_path,
        '-c:v', 'libx264', '-pix_fmt', 'yuv420p',
        '-vf', 'scale=trunc(iw/2)*2:trunc(ih/2)*2',
        '-movflags', '+faststart',
        dst_path,
    ]
    subprocess.run(cmd, check=True)
    print('H.264 video:', dst_path)
    return dst_path

playable_path = reencode_h264(video_path)



9
   frame_idx  track_id           x1          y1           x2          y2  \
0          0        -1    49.490723  369.143921   197.292725  525.930176   
1          0        -1   190.603058  255.723633   306.176819  428.451599   
2          0        -1   903.550293  524.879578  1038.689209  640.000000   
3          0        -1  1024.000000   36.821716  1152.581543  197.118805   
4          0        -1  1970.460205  183.893677  2119.167236  302.265411   

      score  
0  0.467594  
1  0.332962  
2  0.722973  
3  0.458097  
4  0.699982  
уникальных track_id: 1156 | доля -1: 0.157
Saved video to: output/annotated_pose.mp4
Saved frames to: output/frames
H.264 video: output/annotated_pose_h264.mp4
